# 최소 신장 트리(Minimum Spanning Tree, MST)

그래프(특히 무방향 가중치 그래프)에서 **모든 정점**을 연결하되, **간선의 가중치 합이 최소**가 되도록 하는 트리를 **최소 신장 트리(MST)**라고 한다.

즉, **그래프에 있는 모든 정점을 연결**하면서, **최소 비용**(간선 비용 합)을 갖는 부분 그래프를 뜻한다.

## **1. 신장 트리(Spanning Tree)란?**

1. **신장 트리**
    - `n`개의 정점으로 이루어진 연결 그래프에서,
    - **모든 정점**을 포함하고, **사이클이 없는** 트리 형태로 간선들을 **n-1개** 사용해 만든 부분 그래프
    - 예) 정점이 5개인 연결 그래프에서, 4개의 간선만 써서 5개 정점을 잇고, 사이클이 없어야 함
2. **최소 신장 트리(MST)**
    - **무향 가중치 그래프**에서, 그 신장 트리를 이루는 **간선들의 가중치 합이 최소**가 되는 트리
        - 간선에 가중치(비용)가 있을 때, 그 총합이 **최소**가 되는 신장 트리
    - 활용 예
        - **통신망 구축**(가장 적은 비용으로 모든 도시 연결)
        - **전력망**, **도로망** 등

### **1.1 그래프에서 최소 비용 문제**

- **MST**는 “**그래프의 모든 정점을 연결**하면서, **간선들의 총 비용이 최소**”가 되는 문제
    - 🔴**“두 정점 사이의 최소 비용 경로”**와는 다른 문제 (그건 `최단 경로`)🔴
- MST는 **전체 연결**을 목표, 간선 비용 합이 **가장 작도록**

### **1.2 MST 표현**

- 그래프를 어떻게 표현하든(간선 리스트, 인접 리스트, 등), MST 결과는 “간선들의 부분 집합(또는 트리 구조)”가 된다.
- 예) 간선 배열을 두고 **Kruskal**로 MST를 찾거나, 인접 리스트로 **Prim**을 구현

## **2. Kruskal 알고리즘**

<aside>
💡

**간선을 하나씩** 선택하여 MST를 만드는 알고리즘

</aside>

[크루스칼 알고리즘 - 최소 신장 트리](https://www.youtube.com/watch?v=BJZwRWNGFMU)

### **2.1 알고리즘 개념**

1. **간선들을 가중치가 작은 순**(오름차순)으로 정렬
2. **순서대로 간선을 확인**하며, **사이클**이 생기지 않으면 MST에 포함 (Union-Find로 사이클 체크)
3. 모든 정점이 MST에 포함될 때까지 반복

### **2.2 단계적 풀이**

1. 간선을 가중치 작은 순으로 나열
2. 첫 번째 간선 → MST에 추가 (아직 간선이 없으므로 무조건 가능)
3. 두 번째 간선 → MST에 추가 (사이클인지 확인, 아니면 추가)
4. 만약 어떤 간선을 추가하면 **사이클**이 생기면, 그 간선은 무시
5. MST 간선 수가 (정점 수 - 1)이 되면 종료

### **2.3 구현 개요**

- Union-Find(서로소 집합)로 사이클을 빠르게 판별
- 간선 정렬: **`$O(E log E)$`**
- 각 간선을 살펴보는 동안, **Find**로 두 정점이 같은 집합인지 확인 → 다르면 **Union**

### **2.4 특징**

- **그리디 알고리즘** 기반 (가중치가 가장 작은 간선 우선)
- **희소 그래프**(간선이 적은)에서 효율적 (정렬 + Union-Find)
- 음의 가중치도 처리 가능

### 2.5 Kruskal 예시 코드

In [ ]:
def find_set(parent_list, x):
    """
    find_set(x):
    x의 루트(대표) 노드를 찾는 함수.
    '경로 압축(Path Compression)' 기법을 사용하여
    찾은 뒤 부모를 직접 루트로 설정해 탐색 속도를 개선.
    """
    if parent_list[x] != x:
        parent_list[x] = find_set(parent_list, parent_list[x])
    return parent_list[x]


def union(parent_list, rank_list, a, b):
    """
    union(a,b):
    두 원소 a,b를 같은 집합으로 합치는 연산.
    'union by rank'로 랭크(트리높이)가 낮은 쪽을 높은 쪽 밑으로 붙임.
    """
    root_a = find_set(parent_list, a)
    root_b = find_set(parent_list, b)

    if root_a != root_b:
        # 서로 다른 집합이면 합침
        if rank_list[root_a] > rank_list[root_b]:
            parent_list[root_b] = root_a
        elif rank_list[root_a] < rank_list[root_b]:
            parent_list[root_a] = root_b
        else:
            parent_list[root_b] = root_a
            rank_list[root_a] += 1


def kruskal_mst(num_vertices, edges):
    """
    Kruskal 알고리즘으로 MST를 찾는 함수.
    :param num_vertices: 정점(노드) 수
    :param edges: (가중치, 시작정점, 끝정점) 형태의 간선 리스트
    :return: (mst_edges, mst_cost)
             mst_edges: MST에 포함된 간선 리스트
             mst_cost: MST 간선들의 가중치 합(최소 비용)
    """
    # 1) 간선들을 가중치 오름차순 정렬
    edges.sort(key=lambda x: x[0])

    # 2) Union-Find용 parent, rank 배열 초기화
    parent = [i for i in range(num_vertices + 1)]
    rank = [0] * (num_vertices + 1)

    mst_edges = []
    mst_cost = 0

    # 3) 작은 간선부터 하나씩 확인
    for w, s, e in edges:
        # Find로 루트를 비교하여 사이클 확인
        if find_set(parent, s) != find_set(parent, e):
            union(parent, rank, s, e)
            mst_cost += w
            mst_edges.append((s, e, w))
            # 만약 MST에 (num_vertices - 1)개의 간선을 채우면 끝

    return mst_edges, mst_cost


# ---- 사용 예시 ----
n = 5  # 예: 정점이 5개 (1~5번)
edges_info = [
    (1, 1, 2),
    (2, 2, 3),
    (2, 1, 4),
    (3, 3, 4),
    (4, 2, 5),
    (7, 4, 5),
]
result_edges, result_cost = kruskal_mst(n, edges_info)
print("MST 간선:", result_edges)
print("MST 총 비용:", result_cost)

"""
MST 간선: [(1, 2, 1), (2, 3, 2), (1, 4, 2), (2, 5, 4)]
MST 총 비용: 9
"""

- **Union-Find**(서로소 집합) 구조를 사용해 사이클을 빠르게 판별하는 코드
    
    **해설**
    
    1. 간선 목록을 **가중치 오름차순**으로 정렬
    2. **Union-Find**로 사이클 검사 → **Union** 수행
    3. MST 완성 시 **(정점수-1)** 간선이 선택되며, **최소 비용** 계산

## **3. Prim 알고리즘**

<aside>
💡

**임의의 시작 정점**에서 시작해, **MST 집합**을 점차 확장하는 알고리즘 (MST를 만들어 가는)

</aside>

[Prim's algorithm in 2 minutes](https://www.youtube.com/watch?v=cplfcGZmX7I)

### **3.1 알고리즘 개념**

1. **임의의 시작 정점**을 선택 → MST 집합에 포함
2. MST 집합에 있는 정점들과 **연결된 간선 중 최소 가중치**인 간선을 선택 
(이 간선이 이어주는 **새 정점**을 MST에 추가)
3. 해당 정점을 MST에 추가한 뒤, 다시 **MST 내부 정점** 
→ **MST 외부 정점**과 이어지는 최소 간선을 찾음
4. 모든 정점이 MST에 포함될 때까지 반복

### **3.2 단계적 풀이**

1. 시작 노드 = A
2. A와 연결된 간선들 중 가장 작은 비용 간선 → MST에 추가, 그 간선의 반대편 노드를 MST에 편입
3. MST 집합 내 노드가 확장 → MST 내부(2개 노드)에서 바깥 노드로 연결되는 간선 중 가장 작은 비용을 선택
4. 모든 노드가 MST에 포함될 때까지 반복

### **3.3 구현 개요**

- 보통 `우선순위 큐(최소 힙)`를 사용해 “가장 작은 간선”을 빠르게 선택
- 방문(visited) 배열로, MST 내부/외부를 구분
- 그래프 표현은 주로 **인접 리스트** 사용 (간선이 많을 때도 효율적)

### **3.4 특징**

- **그리디 알고리즘** (매 단계, 가중치가 가장 작은 간선을 고름)
- 밀집 그래프(간선이 많은)에서 유리
- 음의 가중치도 일반적으로 가능 
(문헌마다 다를 수 있으나 MST는 보통 무음, 양음 가중치 혼합 가능)

### 3.5 Prim 예시 코드

In [ ]:
import heapq

def prim_mst(num_vertices, adj_list, start=1):
    """
    Prim 알고리즘으로 MST를 구하는 함수.
    :param num_vertices: 정점 수
    :param adj_list: {노드: [(가중치, 인접노드), ...]} 형태의 인접 리스트
    :param start: 시작 노드 (기본 1)
    :return: MST 총 비용
    """

    visited = [False] * (num_vertices + 1)
    priority_queue = []

    # 1) 시작 노드를 MST에 포함했다고 가정, 연결된 간선들 push
    visited[start] = True
    for cost, next_node in adj_list[start]:
        heapq.heappush(priority_queue, (cost, next_node))

    mst_cost = 0
    edges_used = 0  # MST에 추가된 간선 수

    # 2) 큐가 비기 전, 또는 MST 간선이 (n-1)개 될 때까지 반복
    while priority_queue and edges_used < (num_vertices - 1):
        cost, node = heapq.heappop(priority_queue)
        # 이미 방문한 노드면 skip
        if visited[node]:
            continue

        # 미방문 노드이므로 MST에 편입
        visited[node] = True
        mst_cost += cost
        edges_used += 1

        # 해당 노드와 인접한 간선을 우선순위 큐에 추가
        for next_cost, adj_node in adj_list[node]:
            if not visited[adj_node]:
                heapq.heappush(priority_queue, (next_cost, adj_node))

    return mst_cost


# ---- 사용 예시 ----
n = 5
# 인접 리스트: 무방향 그래프, (가중치, 인접노드)
adj_list = {
    1: [(2, 2), (2, 4)],
    2: [(2, 1), (2, 3), (4, 5)],
    3: [(2, 2), (3, 4)],
    4: [(2, 1), (3, 3), (7, 5)],
    5: [(4, 2), (7, 4)],
}

result_cost = prim_mst(n, adj_list, start=1)
print("Prim MST 비용:", result_cost)  # 10

**해설**

1. **시작 노드**에서 인접한 간선을 **우선순위 큐**에 삽입
2. **큐에서 가장 작은 간선**을 pop → 연결되는 노드가 방문 전이면 MST에 추가
3. 새로 방문한 노드의 간선들 다시 큐에 삽입
4. 모든 노드를 MST에 편입할 때까지 반복
5. 최종 MST 간선 비용 합이 `mst_cost`

## **4. Kruskal vs. Prim 정리**

| 구분 | **Kruskal** | **Prim** |
| --- | --- | --- |
| **아이디어** | **간선을 가중치 순**으로 정렬 후, 작은 것부터 하나씩 확인해 MST에 넣기 | **임의 정점**에서 시작, 인접 간선 중 **가장 작은** 간선 선택해 MST 확장 |
| **데이터 구조** | 간선 리스트(가중치로 정렬) + **Union-Find(서로소 집합)** | 정점 방문 여부 + **우선순위 큐(가장 작은 간선)** (또는 배열) |
| **장점** | **희소 그래프**(E가 작을 때)에서 정렬+Union-Find로 효율적음의 가중치 O | **밀집 그래프**(E가 많을 때)에서 인접 리스트+우선순위 큐로 효율적 |
| **단점** | 간선을 정렬해야 함, 동적 그래프 적용 어려움 | 구현 시 우선순위 큐 다뤄야 함(음수 가중치도 가능하지만 구현 주의) |
| **시간 복잡도** | **O(E log E)** (또는 O(E log V)) | **O(E log V)** (우선순위 큐에서 여러 간선을 삽입/팝) |
- 둘 다 **MST**를 찾는다.
- 그래프 구조(간선이 sparse/ dense), 구현 편의 등 조건에 따라 선택할 것

## **5. 마무리**

<aside>
💡

“**모든 노드를 연결하되 비용이 최소**”라는 목표와, 
“Kruskal은 간선이 작은 것부터, Prim은 한 정점에서 출발”이라는 차이를 기억

</aside>

- **최소 신장 트리(MST)**
    - **연결 그래프**에서 모든 정점을 커버하는, 간선 비용 합이 **최소**인 트리를 만드는 문제다.
- **Kruskal 알고리즘**
    - 간선을 **가중치 오름차순**으로 정렬 후, **Union-Find**로 사이클을 피하며 선택
    - **희소 그래프**에 유리, 구현이 비교적 단순
- **Prim 알고리즘**
    - **임의의 정점**에서 시작, **가장 작은 가중치** 간선을 찾으며 MST 확장
    - 한 정점에서 시작해 **최소 간선**을 차례차례 연결
    - **우선순위 큐**(힙) 사용, **밀집 그래프**에 효율적